In [2]:
import requests
import os
import time

### a) URLs

In [3]:
with open(r"C:\Users\aabab\ATP_HW1\wiki_233_diseases_sorted_urls.txt", 'r', encoding='utf-8') as resources:
    urls = [line.strip() for line in resources if line.strip()]

print(f"Total URLs to download: {len(urls)}")
print(urls)

Total URLs to download: 233
['https://en.wikipedia.org/wiki/Acne', 'https://en.wikipedia.org/wiki/Alcoholic_liver_disease', 'https://en.wikipedia.org/wiki/Alcoholism', 'https://en.wikipedia.org/wiki/Allergic_rhinitis', 'https://en.wikipedia.org/wiki/Allergy', "https://en.wikipedia.org/wiki/Alzheimer's_disease", 'https://en.wikipedia.org/wiki/Amnesia', 'https://en.wikipedia.org/wiki/Amyotrophic_lateral_sclerosis', 'https://en.wikipedia.org/wiki/Anemia', 'https://en.wikipedia.org/wiki/Anxiety_disorder', 'https://en.wikipedia.org/wiki/Aphasia', 'https://en.wikipedia.org/wiki/Appendicitis', 'https://en.wikipedia.org/wiki/Arteriosclerosis', 'https://en.wikipedia.org/wiki/Arthritis', 'https://en.wikipedia.org/wiki/Asperger_syndrome', 'https://en.wikipedia.org/wiki/Asthma', 'https://en.wikipedia.org/wiki/Astigmatism', 'https://en.wikipedia.org/wiki/Atherosclerosis', "https://en.wikipedia.org/wiki/Athlete's_foot", 'https://en.wikipedia.org/wiki/Atopic_dermatitis', 'https://en.wikipedia.org/wik

In [4]:
output_dir = r"C:\Users\aabab\ATP_HW1\downloaded_htmls"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"dir generated: {output_dir}")
else:
    print("dir exists")

dir exists


In [5]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}
r = requests.get('https://www.wikipedia.org', headers = headers) 

In [6]:
for i, url in enumerate(urls):
    try:
        # Bring data
        response = requests.get(url, headers = headers)
        response.raise_for_status() # error
        
        response.encoding = 'utf-8'
        
        # generating files name
        file_name = f"{i}.html"
        file_path = os.path.join(output_dir, file_name)
        
        # saving the files
        with open(file_path, 'w', encoding='utf-8') as html_file:
            html_file.write(response.text)
        
        time.sleep(0.1)

    except Exception as e:
        print(f"error ({url}): {e}")

print("Completed")

Completed


## b) Extract

In [11]:
!pip install bs4
from bs4 import BeautifulSoup
import json

In [12]:

input_dir = r"C:\Users\aabab\ATP_HW1\downloaded_htmls"
results = []

for i in range(233):
    file_path = os.path.join(input_dir, f"{i}.html")
    
    if not os.path.exists(file_path):
        continue

    with open(file_path, 'r', encoding='utf-8') as f:
        html_content = f.read()
    
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # 1) Title
    title = soup.title.string if soup.title else "No Title"
    
    # 2) Main Text: <div id="bodyContent">
    body_div = soup.find('div', id='bodyContent')
    main_text = body_div.get_text(strip=True) if body_div else ""
    
    # 3) JSON-LD
    ld = {}
    json_script = soup.find('script', type='application/ld+json')
    
    if json_script:
        try:
            data = json.loads(json_script.string)
            target_keys = ["name", "url", "datePublished", "dateModified", "headline"]
            for key in target_keys:
                ld[key] = data.get(key, None)
        except json.JSONDecodeError:
            pass

    # 4) Merging
    article_data = {
        "title": title,
        "main_text": main_text,
        **ld # merging with ld
    }
    
    results.append(article_data)

print(f"Total {len(results)} data.")

print(results[:1]) # only first data

Total 233 data.
[{'title': 'Acne - Wikipedia', 'main_text': 'From Wikipedia, the free encyclopediaSkin condition characterized by pimplesThis article is about a skin disease common during adolescence. For other acneiform skin diseases, seeAcne (disambiguation).Medical conditionAcneOther namesAcne vulgarisAcne vulgaris in an 18-year-old male duringpubertyPronunciation(/ˈækni/ⓘAK-nee)SpecialtyDermatologySymptomsBlackheads, whiteheads,pimples, oily skin,scarring[1][2]ComplicationsAnxiety, reducedself-esteem,depression,thoughts of suicide[3][4]Usual onsetPuberty[5]Typesblackhead,whitehead,papule,pustule,nodule,cystRisk factorsGenetics[2]Differential diagnosisFolliculitis,rosacea,hidradenitis suppurativa,miliaria,perioral dermatitis[6]TreatmentLifestyle changes, medications, medical procedures[7][8]MedicationAzelaic acid,benzoyl peroxide,salicylic acid,antibiotics,birth control pills,co-cyprindiol,retinoids,isotretinoin[8]Frequency633 million affected (2015)[9]Acne, also known asacne vulgar

## c) Create a string of wikipedia main text

In [10]:
# 1. Main text to list
all_texts = [data['main_text'] for data in results]

# 2. Concatenate all text with a space
combined_text = " ".join(all_texts)

# 3. Saving output
output_txt_path = r"C:\Users\aabab\ATP_HW1\all_diseases_text.txt"

with open(output_txt_path, 'w', encoding='utf-8') as f:
    f.write(combined_text)

## d) Create a pandas DataFrame

In [13]:
import pandas as pd

# 1. results to DataFrame
articles = pd.DataFrame(results)

# 2. 'title' to index
articles.set_index('title', inplace=True)

# 3. output dataframe
print("--- DataFrame Articles (Head) ---")
print(articles.head())

# 4. Search not "dateModified" in JSON-LD
no_date_modified = articles[articles['dateModified'].isnull()]

# 5. output
print("\n--- Articles without 'dateModified' information ---")
print(no_date_modified.index.tolist())

--- DataFrame Articles (Head) ---
                                                                             main_text  \
title                                                                                    
Acne - Wikipedia                     From Wikipedia, the free encyclopediaSkin cond...   
Alcoholic liver disease - Wikipedia  From Wikipedia, the free encyclopediaMedical c...   
Alcoholism - Wikipedia               From Wikipedia, the free encyclopediaProblemat...   
Allergic rhinitis - Wikipedia        From Wikipedia, the free encyclopediaNasal inf...   
Allergy - Wikipedia                  From Wikipedia, the free encyclopediaImmune sy...   

                                                        name  \
title                                                          
Acne - Wikipedia                                        Acne   
Alcoholic liver disease - Wikipedia  Alcoholic liver disease   
Alcoholism - Wikipedia                            Alcoholism   
Allergic rhinit

These 4 diseases don't have 'dateModified' information.
['Breast cancer', 'Dementia', 'Diabetes', 'Eating disorder']

##  e) Sort the DataFrame based on the length of the 'headline' column

In [20]:
# 1. Counting 'headline' length
sorted_articles = articles.sort_values(by='headline', key=lambda x: x.str.len(), ascending=True)

# 2. first 5 head rows
first_5_rows = sorted_articles.head(5)

# 3. output
print("\n--- Wikipedia Articles' Titles (Top 5) ---")
print(first_5_rows.index.tolist())


print("--- First 5 rows: Title and Headline Content ---")
print(first_5_rows[['headline']])




--- Wikipedia Articles' Titles (Top 5) ---
['Meningism - Wikipedia', 'Venous ulcer - Wikipedia', 'Sleep paralysis - Wikipedia', 'Periodontal disease - Wikipedia', "Athlete's foot - Wikipedia"]
--- First 5 rows: Title and Headline Content ---
                                     headline
title                                        
Meningism - Wikipedia                 symptom
Venous ulcer - Wikipedia              disease
Sleep paralysis - Wikipedia        phenomenon
Periodontal disease - Wikipedia   gum disease
Athlete's foot - Wikipedia       foot disease


--- the first 5 rows of Wikipedia Articles' Titles (Top 5) ---

['Meningism - Wikipedia', 'Venous ulcer - Wikipedia', 'Sleep paralysis - Wikipedia', 'Periodontal disease - Wikipedia', "Athlete's foot - Wikipedia"]

## f) Store the DataFrame in a local sqlite3 database

In [21]:
import sqlite3

In [22]:
# 1. setting db path
db_path = r"C:\Users\aabab\ATP_HW1\wiki_diseases.db"
conn = sqlite3.connect(db_path)

# 2. Store the DataFrame in a local sqlite3 database
articles.to_sql(name="articles", con=conn, index_label="title", if_exists="replace")

# 3. Closing the connection
conn.close()

print(f"Completed: {db_path}")

Completed: C:\Users\aabab\ATP_HW1\wiki_diseases.db


In [23]:
# 1. Connecting
conn = sqlite3.connect("wiki_diseases.db")

# 2. Reading SQL
df_check = pd.read_sql_query("SELECT * FROM articles", conn)

# 3. Closing
conn.close()

# 4. Checking the data
print(f"Stored data: {len(df_check)}")
df_check.head()

Stored data: 233


,title,main_text,name,url,datePublished,dateModified,headline
0,Acne - Wikipedia,"From Wikipedia, the free encyclopediaSkin cond...",Acne,https://en.wikipedia.org/wiki/Acne,2002-08-22T01:27:34Z,2026-01-25T06:17:40Z,skin condition characterized by pimples
1,Alcoholic liver disease - Wikipedia,"From Wikipedia, the free encyclopediaMedical c...",Alcoholic liver disease,https://en.wikipedia.org/wiki/Alcoholic_liver_...,2003-08-29T06:49:59Z,2026-01-14T21:46:13Z,medical condition
2,Alcoholism - Wikipedia,"From Wikipedia, the free encyclopediaProblemat...",Alcoholism,https://en.wikipedia.org/wiki/Alcoholism,2001-12-20T15:13:43Z,2026-01-12T21:39:35Z,loss of control over the quantity of consumed ...
3,Allergic rhinitis - Wikipedia,"From Wikipedia, the free encyclopediaNasal inf...",Allergic rhinitis,https://en.wikipedia.org/wiki/Allergic_rhinitis,2003-12-24T04:16:09Z,2026-02-02T02:09:58Z,inflammation in nose occurs when immune system...
4,Allergy - Wikipedia,"From Wikipedia, the free encyclopediaImmune sy...",Allergy,https://en.wikipedia.org/wiki/Allergy,2002-06-07T10:03:08Z,2026-01-24T18:41:48Z,immune system response to a substance that mos...
